Handling imbalanced data

In [17]:
import mlflow
mlflow.set_tracking_uri('http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/')

In [19]:
mlflow.set_experiment("Exp 4- handling imbalanced data")

2026/03/16 11:59:32 INFO mlflow.tracking.fluent: Experiment with name 'Exp 4- handling imbalanced data' does not exist. Creating a new experiment.


<Experiment: artifact_location='s3://mlflow-bucket-1807077/24', creation_time=1773640772579, experiment_id='24', last_update_time=1773640772579, lifecycle_stage='active', name='Exp 4- handling imbalanced data', tags={}>

In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [21]:
df=pd.read_csv("processed_data.csv")
df['clean_comment'] = df['clean_comment'].fillna('')
df = df[['clean_comment', 'category']]
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [22]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [24]:
# Step 1: Function to run the experiment
def run_imbalanced_experiment(imbalance_method):

    ngram_range = (1, 2)  # Bigram setting
    max_features = 10000  # Set max_features to 1000 for TF-IDF

    # Step 4: Train-test split before vectorization and resampling
    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_comment'],
        df['category'],
        test_size=0.2,
        random_state=42
    )

    # Step 2: Vectorization using TF-IDF, fit on training data only
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X_train_vec = vectorizer.fit_transform(X_train)   # Fit on training data
    X_test_vec = vectorizer.transform(X_test)         # Transform test data

    # Step 3: Handle class imbalance based on the selected method
    if imbalance_method == 'class_weights':
        class_weight = 'balanced'
    else:
        class_weight = None  # Do not apply class_weight if using resampling

        # Resampling Techniques (only apply to the training set)
        if imbalance_method == 'oversampling':
            smote = SMOTE(random_state=42)
            X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)

        elif imbalance_method == 'adasyn':
            adasyn = ADASYN(random_state=42)
            X_train_vec, y_train = adasyn.fit_resample(X_train_vec, y_train)

        elif imbalance_method == 'undersampling':
            rus = RandomUnderSampler(random_state=42)
            X_train_vec, y_train = rus.fit_resample(X_train_vec, y_train)

        elif imbalance_method == 'smote':
            smote_enn = SMOTEENN(random_state=42)
            X_train_vec, y_train = smote_enn.fit_resample(X_train_vec, y_train)

    # Step 5: Define and train a Random Forest model
    with mlflow.start_run() as run:

        # Set tags
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)

        # Initialize and train model
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            class_weight=class_weight
        )

        model.fit(X_train_vec, y_train)

        # Step 6: Make predictions
        y_pred = model.predict(X_test_vec)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)

        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Bigrams, Imbalance={imbalance_method}")

        confusion_matrix_filename = f"confusion_matrix_{imbalance_method}.png"
        plt.savefig(confusion_matrix_filename)

        mlflow.log_artifact(confusion_matrix_filename)
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(
            model,
            f"random_forest_model_tfidf_trigrams_imbalance_{imbalance_method}"
        )


# Step 7: Run experiments for different imbalance methods
imbalance_methods = [
    'class_weights',
    'oversampling',
    'adasyn',
    'undersampling',
    'smote'
]

for method in imbalance_methods:
    run_imbalanced_experiment(method)

/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)
2026/03/16 12:00:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/16 12:00:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigrams at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24/runs/8574ab908c5842168796d0ab0a9c9483
🧪 View experiment at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24


/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
2026/03/16 12:01:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/16 12:02:07 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigrams at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24/runs/7bb49a3046d14551af75948144ecab2a
🧪 View experiment at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24


/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
2026/03/16 12:03:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/16 12:03:53 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Imbalance_adasyn_RandomForest_TFIDF_Trigrams at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24/runs/855d003d65d440629efae48b426ba61d
🧪 View experiment at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24


/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(
2026/03/16 12:04:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/16 12:05:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the 

🏃 View run Imbalance_undersampling_RandomForest_TFIDF_Trigrams at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24/runs/9acb61ddd7f24e54a985c4b28c9b77d5
🧪 View experiment at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24


/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/rakibmahmud/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
2026/03/16 12:06:13 WA

🏃 View run Imbalance_smote_RandomForest_TFIDF_Trigrams at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24/runs/fb899855e2564840b1d1b5c877fe5fb6
🧪 View experiment at: http://ec2-51-20-42-109.eu-north-1.compute.amazonaws.com:5000/#/experiments/24
